# ACM — Banking Dataset (Marketing Targets)
**Lead University · Minería de Datos · Tarea 4**

Análisis de Correspondencias Múltiples sobre datos de campañas de marketing bancario.
Objetivo: explorar asociaciones entre el perfil del cliente (trabajo, educación, estado civil) y el resultado de la campaña para identificar segmentos de interés.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from prince import MCA
from IPython.display import display

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110

RNG = 42
print('Librerías cargadas.')

---
## Parte 1 — Preparación de datos

### 1. Carga y exploración

In [ ]:
df_raw = pd.read_csv('datos/Banking_Dataset.csv', sep=';')
print(f'Registros: {df_raw.shape[0]:,} | Variables: {df_raw.shape[1]}')
print()
print(df_raw.dtypes)
print()
df_raw.head()

In [ ]:
print('Valores faltantes:')
print(df_raw.isnull().sum())
print()
print('Valores únicos por variable categórica:')
for c in df_raw.select_dtypes(include='object').columns:
    print(f'  {c}: {df_raw[c].nunique()} -> {df_raw[c].unique().tolist()}')

### 2. Limpieza

- No hay valores faltantes (NaN). Sin embargo, algunas variables tienen la categoría `unknown` que se conserva como modalidad válida (refleja información no disponible en la campaña).
- Se eliminan las variables numéricas de contacto que no son relevantes para el perfil del cliente (`day`, `duration`, `campaign`, `pdays`, `previous`).
- Se discretizan `age` y `balance` para incluirlas en el ACM.

In [ ]:
df = df_raw.copy()
print(f'Registros iniciales: {len(df):,}')
print(f'Sin NaN: {df.dropna().shape[0]:,}')

### 3-4. Selección de variables categóricas y discretización

In [ ]:
# Variables categóricas naturales
cat_cols = ['job', 'marital', 'education', 'default', 'housing', 'loan',
            'contact', 'poutcome', 'y']

# Discretización de numéricas relevantes
df['age_cat'] = pd.cut(df['age'], bins=[0, 30, 40, 50, 60, 100],
                        labels=['joven', 'adulto_joven', 'adulto', 'adulto_mayor', 'senior'])
df['balance_cat'] = pd.qcut(df['balance'].clip(lower=0), q=4,
                             labels=['bajo', 'medio_bajo', 'medio_alto', 'alto'],
                             duplicates='drop')

cat_cols_full = cat_cols + ['age_cat', 'balance_cat']

df_acm = df[cat_cols_full].dropna().astype(str).reset_index(drop=True)
print(f'Registros para ACM: {len(df_acm):,} | Variables: {len(cat_cols_full)}')
print()
print('Modalidades por variable:')
for c in cat_cols_full:
    print(f'  {c}: {df_acm[c].nunique()} categorías')
print(f'\nTotal de modalidades: {sum(df_acm[c].nunique() for c in cat_cols_full)}')

In [ ]:
# Muestra para eficiencia computacional
N_SAMPLE = 15000
df_sample = df_acm.sample(n=N_SAMPLE, random_state=RNG).reset_index(drop=True)
print(f'Muestra para ACM: {len(df_sample):,} registros')

---
## Parte 2 — Aplicación del ACM

### 6. ¿Qué es el ACM?

El **Análisis de Correspondencias Múltiples (ACM)** es una extensión del ACS para analizar simultáneamente las asociaciones entre más de dos variables categóricas. Opera sobre la tabla disyuntiva completa (codificación 0/1 de todas las modalidades) y aplica una descomposición en valores singulares (SVD) ponderada por chi-cuadrado.

**Objetivo:** Reducir la dimensionalidad de datos categóricos preservando las asociaciones entre modalidades. Permite visualizar qué categorías tienden a aparecer juntas y qué individuos comparten perfiles similares.

**¿Por qué es apropiado para este dataset?** Las variables principales del marketing bancario son categóricas (tipo de trabajo, estado civil, nivel educativo, resultado de campaña). El ACM permite detectar qué perfiles de cliente se asocian con la suscripción al producto (`y=yes`), sin imponer un modelo supervisado.

### 5. Ajuste del ACM

In [ ]:
N_COMP = 10
mca = MCA(n_components=N_COMP, random_state=RNG)
mca.fit(df_sample)

print(f'Inercia total: {mca.total_inertia_:.4f}')
print(f'Componentes ajustados: {N_COMP}')

### 7. Tabla de inercia explicada por componente

In [ ]:
eigenvalues = mca.eigenvalues_
inercia_pct = 100 * eigenvalues / mca.total_inertia_
inercia_acum = np.cumsum(inercia_pct)

tabla_inercia = pd.DataFrame({
    'Dimensión': range(1, N_COMP + 1),
    'Autovalor': eigenvalues,
    '% Inercia': inercia_pct,
    '% Acumulada': inercia_acum
}).round(4)

display(tabla_inercia)

### 8. Scree plot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(1, N_COMP + 1)

ax.bar(x, inercia_pct, color='coral', alpha=0.85, label='% inercia')
ax2 = ax.twinx()
ax2.plot(x, inercia_acum, 'o-', color='darkred', lw=2, markersize=5, label='% acumulada')
ax2.axhline(80, ls='--', color='gray', lw=0.8, label='80%')

ax.set_xlabel('Dimensión')
ax.set_ylabel('% Inercia')
ax2.set_ylabel('% Inercia acumulada')
ax.set_xticks(x)
ax.set_title('Scree Plot — ACM Banking')
ax2.legend(loc='center right')
plt.tight_layout()
plt.show()

### 9. Elección de componentes

Al igual que en el dataset de vuelos, la inercia en ACM se distribuye uniformemente entre las dimensiones. Se seleccionan las **primeras 3 dimensiones** por el criterio del codo (las que muestran mayor salto relativo antes de estabilizarse). Estas capturan los contrastes principales entre perfiles de clientes.

---
## Parte 3 — Interpretación del espacio factorial

### 10. Biplot del ACM

In [ ]:
coords_ind = mca.row_coordinates(df_sample)
coords_mod = mca.column_coordinates(df_sample)

fig, ax = plt.subplots(figsize=(12, 9))

# Individuos
n_plot_ind = 3000
idx_plot = np.random.default_rng(RNG).choice(len(coords_ind), size=n_plot_ind, replace=False)
ax.scatter(coords_ind.iloc[idx_plot, 0], coords_ind.iloc[idx_plot, 1],
           s=8, alpha=0.2, c='coral', linewidths=0, label='Individuos')

# Modalidades
ax.scatter(coords_mod.iloc[:, 0], coords_mod.iloc[:, 1],
           s=60, marker='s', c='darkred', edgecolors='white', linewidths=0.5,
           zorder=3, label='Modalidades')

for idx, row in coords_mod.iterrows():
    ax.annotate(idx, (row.iloc[0], row.iloc[1]), fontsize=7, color='darkred',
                ha='left', va='bottom')

ax.axhline(0, c='gray', lw=0.5)
ax.axvline(0, c='gray', lw=0.5)
ax.set_xlabel(f'Dim 1 ({inercia_pct[0]:.1f}% inercia)')
ax.set_ylabel(f'Dim 2 ({inercia_pct[1]:.1f}% inercia)')
ax.set_title('Biplot ACM — Banking (individuos + modalidades)')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

### 11. Interpretación visual del biplot

In [ ]:
# Mapa de modalidades coloreado por variable
fig, ax = plt.subplots(figsize=(13, 10))

colors = plt.cm.tab10(np.linspace(0, 1, len(cat_cols_full)))
color_map = {col: colors[i] for i, col in enumerate(cat_cols_full)}

for var in cat_cols_full:
    var_mods = [idx for idx in coords_mod.index if idx.startswith(var)]
    if var_mods:
        subset = coords_mod.loc[var_mods]
        ax.scatter(subset.iloc[:, 0], subset.iloc[:, 1],
                   s=80, marker='s', c=[color_map[var]], edgecolors='white',
                   linewidths=0.5, zorder=3, label=var)
        for idx, row in subset.iterrows():
            ax.annotate(idx, (row.iloc[0], row.iloc[1]), fontsize=7,
                        ha='left', va='bottom')

ax.axhline(0, c='gray', lw=0.5)
ax.axvline(0, c='gray', lw=0.5)
ax.set_xlabel(f'Dim 1 ({inercia_pct[0]:.1f}% inercia)')
ax.set_ylabel(f'Dim 2 ({inercia_pct[1]:.1f}% inercia)')
ax.set_title('Mapa de modalidades — ACM Banking')
ax.legend(loc='best', fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

**Interpretación del biplot:**

*(Se completa tras ejecutar el notebook.)*

Puntos a observar:
- **Asociaciones esperadas:** `poutcome_success` debería estar cerca de `y_yes` (si la campaña previa fue exitosa, es más probable que suscriba de nuevo). `retired` y `senior` posiblemente se asocien con `y_yes` (mayor disponibilidad/interés en productos de ahorro).
- **Grupos visibles:** Clientes con `default_yes` probablemente se separen como un grupo atípico.
- **Cerca del origen:** Las modalidades mayoritarias (`housing_yes`, `loan_no`, `default_no`) estarán cerca del centro por ser el perfil promedio.

### 12. Contribuciones a las primeras dimensiones

In [ ]:
contrib = mca.column_contributions_
contrib_top = contrib.iloc[:, :3].copy()
contrib_top.columns = ['Dim 1 (%)', 'Dim 2 (%)', 'Dim 3 (%)']
contrib_top = (contrib_top * 100).round(2)

print('Top 15 modalidades por contribución a Dim 1:')
display(contrib_top.sort_values('Dim 1 (%)', ascending=False).head(15))
print()
print('Top 15 modalidades por contribución a Dim 2:')
display(contrib_top.sort_values('Dim 2 (%)', ascending=False).head(15))

In [ ]:
# Contribución agregada por variable
contrib_var = pd.DataFrame(index=cat_cols_full, columns=['Dim 1 (%)', 'Dim 2 (%)'])
for var in cat_cols_full:
    var_mods = [idx for idx in contrib.index if idx.startswith(var)]
    if var_mods:
        contrib_var.loc[var, 'Dim 1 (%)'] = (contrib.loc[var_mods, 0] * 100).sum()
        contrib_var.loc[var, 'Dim 2 (%)'] = (contrib.loc[var_mods, 1] * 100).sum()

contrib_var = contrib_var.astype(float).round(2)
print('Contribución agregada por variable:')
display(contrib_var.sort_values('Dim 1 (%)', ascending=False))

### 13. Perfiles similares en el espacio reducido

In [ ]:
# Colorear por resultado de campaña (y)
fig, ax = plt.subplots(figsize=(10, 7))

palette = {'no': 'steelblue', 'yes': 'coral'}
for resultado, color in palette.items():
    mask = df_sample['y'] == resultado
    ax.scatter(coords_ind.loc[mask, 0], coords_ind.loc[mask, 1],
               s=8, alpha=0.3, c=color, linewidths=0, label=f'y={resultado}')

ax.axhline(0, c='gray', lw=0.5)
ax.axvline(0, c='gray', lw=0.5)
ax.set_xlabel(f'Dim 1 ({inercia_pct[0]:.1f}% inercia)')
ax.set_ylabel(f'Dim 2 ({inercia_pct[1]:.1f}% inercia)')
ax.set_title('Plano ACM — Coloreado por suscripción (y)')
ax.legend()
plt.tight_layout()
plt.show()

# Centroides por y
print('Centroides por resultado de campaña:')
coords_y = coords_ind.iloc[:, :2].copy()
coords_y['y'] = df_sample['y'].values
print(coords_y.groupby('y')[[0, 1]].mean().round(3))

In [ ]:
# Perfiles por tipo de trabajo
fig, ax = plt.subplots(figsize=(10, 7))

jobs = sorted(df_sample['job'].unique())
colors_job = plt.cm.tab20(np.linspace(0, 1, len(jobs)))

for i, job in enumerate(jobs):
    mask = df_sample['job'] == job
    ax.scatter(coords_ind.loc[mask, 0], coords_ind.loc[mask, 1],
               s=10, alpha=0.3, c=[colors_job[i]], linewidths=0, label=job)

ax.axhline(0, c='gray', lw=0.5)
ax.axvline(0, c='gray', lw=0.5)
ax.set_xlabel(f'Dim 1 ({inercia_pct[0]:.1f}% inercia)')
ax.set_ylabel(f'Dim 2 ({inercia_pct[1]:.1f}% inercia)')
ax.set_title('Plano ACM — Coloreado por tipo de trabajo')
ax.legend(loc='best', fontsize=7, ncol=2)
plt.tight_layout()
plt.show()

**Análisis de perfiles:**

*(Se completa tras ejecutar.)*

### Cos² — Calidad de representación

In [ ]:
cos2 = mca.column_cosine_similarities(df_sample)
cos2_plano = cos2.iloc[:, :2].copy()
cos2_plano.columns = ['cos² Dim1', 'cos² Dim2']
cos2_plano['cos² total (1+2)'] = cos2_plano.sum(axis=1)

print('Modalidades mejor representadas en el plano (top 15):')
display(cos2_plano.sort_values('cos² total (1+2)', ascending=False).head(15).round(4))

---
## Parte 4 — Comparación y discusión

### 14. Comparación entre datasets

| Criterio | Vuelos | Banking |
|----------|--------|---------|
| **Asociaciones más claras** | *(completar tras ejecutar)* | *(completar tras ejecutar)* |
| **Mayor dispersión** | *(completar)* | *(completar)* |
| **Más fácil de interpretar** | *(completar)* | *(completar)* |

**Discusión esperada:**
- El dataset de vuelos tiene variables con pocas categorías y relaciones de mercado claras (clase ↔ precio ↔ aerolínea), por lo que debería mostrar asociaciones más nítidas.
- El dataset bancario tiene más variables y modalidades (12 tipos de trabajo + unknown), lo que dispersa la inercia entre más dimensiones y hace el mapa más complejo.
- La interpretabilidad depende de cuán bien definidos estén los perfiles: en vuelos hay dos perfiles claros (economy vs business); en banking los perfiles son más graduales.

### 15. Comparación ACM vs PCA

| Aspecto | PCA | ACM |
|---------|-----|-----|
| **Tipo de variables** | Numéricas continuas | Categóricas nominales/ordinales |
| **Métrica de distancia** | Euclidiana (correlaciones de Pearson) | Chi-cuadrado (desviaciones de independencia) |
| **Entrada** | Matriz numérica (n × p) | Tabla disyuntiva completa (0/1) |
| **Qué maximiza** | Varianza explicada | Inercia (variabilidad respecto a independencia) |
| **Interpretación** | Correlaciones entre variables, ejes como combinaciones lineales | Asociaciones entre modalidades, proximidad = co-ocurrencia |
| **Casos de uso** | Datos de sensores, financieros, mediciones | Encuestas, marketing, segmentación, datos cualitativos |
| **Limitación** | No aplica directamente a categóricas | Inercia total se diluye con muchas modalidades |

**Clave:** Si todas las variables son categóricas, el ACM es el enfoque principal. Aplicar PCA sobre dummies (one-hot) es una alternativa pero distorsiona la geometría porque trata indicadores binarios como continuos y mezcla efectos de frecuencia con estructura de asociación.

### 16. Perspectiva de negocio

**¿Cómo podría utilizarse este análisis en una aerolínea?**

- **Segmentación de mercado:** El ACM identifica perfiles de viajeros (business vs economy, rutas cortas vs largas, compra anticipada vs último momento). Esto permite personalizar ofertas y comunicación por segmento.
- **Optimización de rutas:** Las asociaciones entre ciudades, horarios y aerolíneas revelan nichos desatendidos o rutas donde una aerolínea domina un perfil particular.
- **Pricing dinámico:** Entender qué combinaciones de factores se asocian con precios premium ayuda a calibrar estrategias de precios.

**¿Cómo podría utilizarse en marketing bancario?**

- **Targeting de campañas:** El ACM revela qué perfiles demográficos (edad, trabajo, educación) se asocian con suscripción exitosa (`y=yes`). Esto permite enfocar el esfuerzo de contacto en segmentos con mayor probabilidad de conversión.
- **Personalización del mensaje:** Diferentes clusters en el mapa factorial sugieren diferentes motivaciones; el mensaje puede adaptarse (seguridad financiera para mayores, inversión para profesionales jóvenes).
- **Exclusión inteligente:** Identificar perfiles que sistemáticamente rechazan (`y=no` + muchos contactos previos fallidos) permite reducir costos de campaña sin perder oportunidades.